# HMR-GNN — Run experiments on a free Colab GPU

**Heterophily-Aware Multi-Relational GNN for Robust Fake Account Detection.**

This notebook clones the repo (code + MGTAB data), installs dependencies, runs the
full experiment suite (baselines + hyperparameter tuning + ablation) on the **bot**
(fake-account) task across 5 seeds, and downloads all paper tables.

### Before you start
1. Go to **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**, then Save.
2. Run each cell top to bottom (Shift+Enter).

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Clone the repository (code + data)

In [ ]:
import os
if not os.path.isdir('/content/HMR-GNN'):
    !git clone https://github.com/Vincentfernandes17/HMR-GNN.git /content/HMR-GNN
%cd /content/HMR-GNN
!git pull
# sanity check that the data shipped with the repo
assert os.path.exists('data/MGTAB/features.pt'), 'MGTAB data missing from repo!'
print('Repo ready. Data files:', os.listdir('data/MGTAB'))

## 3. Install dependencies
Colab already ships a GPU build of PyTorch, so we only install the rest to avoid
downgrading torch to a CPU build.

In [ ]:
!pip install -q "scikit-learn>=1.3" "scipy>=1.10" "pandas>=2.0" "numpy>=1.24"
import sklearn, scipy, pandas, numpy
print('sklearn', sklearn.__version__, '| scipy', scipy.__version__, '| pandas', pandas.__version__)

## 4. Quick smoke test (about 1 minute)
Runs 3 models for 2 epochs to confirm everything works before the long run.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/run_experiments.py --mode smoke --task bot --epochs 2 --patience 2 --quiet
print('\nSmoke test finished OK.')

## 5. Full experiment suite (the real run)
Runs all baselines (MLP, GCN, GraphSAGE, GAT, RGCN, DirRGCN, HMR, HMR-Full),
hyperparameter tuning, and the ablation study on the bot task across 5 seeds.

On a T4 GPU this typically takes **roughly 1-3 hours**. Keep the tab open.

*If you are short on time, lower `--trials` (e.g. 6) or `--seeds 42,52,62`.*

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/run_experiments.py \
    --mode all \
    --task bot \
    --seeds 42,52,62,72,82 \
    --epochs 500 \
    --patience 50 \
    --trials 12 \
    --quiet

## 6. Preview the key paper tables

In [ ]:
import pandas as pd, glob, os
def show(path, title):
    if os.path.exists(path):
        print('\n===', title, '===')
        display(pd.read_csv(path))
    else:
        print('(missing)', path)
show('results/bot/tables/baseline_comparison.csv', 'BASELINE COMPARISON (mean +/- std)')
show('results/bot/tables/ablation_comparison.csv', 'ABLATION STUDY')
show('results/bot/tables/significance_tests.csv', 'SIGNIFICANCE TESTS')
show('results/bot/tables/best_hyperparameters.json'.replace('.json', '.csv') if False else 'results/bot/tables/hyperparameter_tuning.csv', 'HYPERPARAMETER TUNING')

## 7. Download all results as a zip
Contains every CSV, Markdown, and LaTeX table plus per-run artifacts.

In [ ]:
import shutil
shutil.make_archive('/content/hmrgnn_results', 'zip', 'results')
from google.colab import files
files.download('/content/hmrgnn_results.zip')
print('Download started: hmrgnn_results.zip')